# NeuroVerse Fusion Method Comparison

## Objective
Compare **3 fusion methods** across **all model combinations** to find the most robust approach for multi-modal AD/PD detection.

### Methods Under Test
| # | Method | Approach |
|---|--------|----------|
| 1 | **Weighted Average** | AUC-proportional weighted sum of risk scores |
| 2 | **Bayesian Sequential Update** | Likelihood-ratio updates starting from epidemiological priors |
| 3 | **Dempster-Shafer Evidence Theory** | BPA mass functions + Dempster's combination rule |

### Models
- **AD detectors:** CDT (AUC 0.989), TMT (AUC 0.857), Speech-AD (AUC 0.967)
- **PD detectors:** Spiral (AUC 0.955), Meander (AUC 0.971), Speech-PD (AUC 0.922)

### Combination Levels
Individual → Pairs → Triples → All available

In [1]:
# ============================================================
# Section 1: Import Required Libraries
# ============================================================
import sys, os
import numpy as np
import json
from collections import defaultdict
from itertools import combinations
from dataclasses import dataclass, asdict

# Add backend to path so we can import our fusion engine
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.path.abspath("."))

from app.ml.fusion.multimodel_fusion import (
    NeuroVerseFusionEngine,
    FusionResult,
    TaskFusionResult,
    ComprehensiveFusionResult,
    MODEL_AUCS,
    PRIOR_AD,
    PRIOR_PD,
    _weighted_average,
    _bayesian_fusion,
    _dempster_shafer,
)

engine = NeuroVerseFusionEngine()
print("✅ Fusion engine loaded")
print(f"   AD models: {list(MODEL_AUCS['ad'].keys())} → AUCs: {MODEL_AUCS['ad']}")
print(f"   PD models: {list(MODEL_AUCS['pd'].keys())} → AUCs: {MODEL_AUCS['pd']}")
print(f"   Priors:    AD={PRIOR_AD}, PD={PRIOR_PD}")
print(f"   Methods:   {list(engine.METHODS.keys())}")

✅ Fusion engine loaded
   AD models: ['cdt', 'tmt', 'speech'] → AUCs: {'cdt': 0.989, 'tmt': 0.857, 'speech': 0.967}
   PD models: ['spiral', 'meander', 'speech'] → AUCs: {'spiral': 0.955, 'meander': 0.971, 'speech': 0.922}
   Priors:    AD=0.1, PD=0.015
   Methods:   ['weighted_avg', 'bayesian', 'dempster_shafer']


## Section 2: Define Test Patient Scenarios

We test **6 clinically realistic patient profiles** — from clearly healthy to clearly diseased — to stress-test each fusion method's behaviour across the full spectrum.

In [2]:
# ============================================================
# Section 2: Define Test Patient Scenarios
# ============================================================

SCENARIOS = {
    # --- Scenario 1: Clearly healthy across all tests ---
    "Healthy (all tests)": {
        "cdt_ad": 0.08, "tmt_ad": 0.12, "speech_ad": 0.15,
        "spiral_pd": 0.10, "meander_pd": 0.07, "speech_pd": 0.09,
    },
    
    # --- Scenario 2: Clear AD patient ---
    "Clear AD": {
        "cdt_ad": 0.92, "tmt_ad": 0.78, "speech_ad": 0.85,
        "spiral_pd": 0.12, "meander_pd": 0.09, "speech_pd": 0.11,
    },
    
    # --- Scenario 3: Clear PD patient ---
    "Clear PD": {
        "cdt_ad": 0.10, "tmt_ad": 0.15, "speech_ad": 0.18,
        "spiral_pd": 0.88, "meander_pd": 0.91, "speech_pd": 0.79,
    },
    
    # --- Scenario 4: Borderline / Ambiguous ---
    "Borderline": {
        "cdt_ad": 0.48, "tmt_ad": 0.52, "speech_ad": 0.45,
        "spiral_pd": 0.42, "meander_pd": 0.55, "speech_pd": 0.38,
    },
    
    # --- Scenario 5: Conflicting models (CDT says AD, TMT disagrees) ---
    "Conflicting Models": {
        "cdt_ad": 0.90, "tmt_ad": 0.20, "speech_ad": 0.60,
        "spiral_pd": 0.75, "meander_pd": 0.25, "speech_pd": 0.50,
    },
    
    # --- Scenario 6: Partial data (only CDT + Spiral available) ---
    "Partial Data (CDT+Spiral)": {
        "cdt_ad": 0.85,
        "spiral_pd": 0.70,
    },
    
    # --- Scenario 7: Single model only ---
    "Single Model (CDT only)": {
        "cdt_ad": 0.82,
    },
    
    # --- Scenario 8: High AD + moderate PD (comorbidity-like) ---
    "Mixed AD+PD signals": {
        "cdt_ad": 0.78, "tmt_ad": 0.65, "speech_ad": 0.72,
        "spiral_pd": 0.68, "meander_pd": 0.60, "speech_pd": 0.55,
    },
}

print(f"Defined {len(SCENARIOS)} test scenarios:")
for name, scores in SCENARIOS.items():
    models = [k.rsplit('_', 1)[0] for k in scores]
    print(f"  • {name:30s} → {len(scores)} scores ({', '.join(models)})")

Defined 8 test scenarios:
  • Healthy (all tests)            → 6 scores (cdt, tmt, speech, spiral, meander, speech)
  • Clear AD                       → 6 scores (cdt, tmt, speech, spiral, meander, speech)
  • Clear PD                       → 6 scores (cdt, tmt, speech, spiral, meander, speech)
  • Borderline                     → 6 scores (cdt, tmt, speech, spiral, meander, speech)
  • Conflicting Models             → 6 scores (cdt, tmt, speech, spiral, meander, speech)
  • Partial Data (CDT+Spiral)      → 2 scores (cdt, spiral)
  • Single Model (CDT only)        → 1 scores (cdt)
  • Mixed AD+PD signals            → 6 scores (cdt, tmt, speech, spiral, meander, speech)


## Section 3: Run Comprehensive Fusion on All Scenarios

For each scenario, run the full engine (`fuse()`) which tests **3 methods × all model combinations** and picks the best via robustness scoring.

In [3]:
# ============================================================
# Section 3: Run Comprehensive Fusion on All Scenarios
# ============================================================

all_results = {}

for name, scores in SCENARIOS.items():
    result = engine.fuse(scores)
    all_results[name] = result
    
    print(f"\n{'='*70}")
    print(f"  {name}")
    print(f"{'='*70}")
    print(f"  Final: {result.final_classification} "
          f"(confidence={result.final_confidence:.4f})")
    print(f"  AD risk: {result.final_ad_risk:.4f}  |  PD risk: {result.final_pd_risk:.4f}")
    print(f"  AD best method: {result.ad.best.method} "
          f"(models={result.ad.best.models_used}, "
          f"risk={result.ad.best.risk:.4f})")
    print(f"  PD best method: {result.pd.best.method} "
          f"(models={result.pd.best.models_used}, "
          f"risk={result.pd.best.risk:.4f})")
    print(f"  AD combos tested: {len(result.ad.all_combinations)}")
    print(f"  PD combos tested: {len(result.pd.all_combinations)}")

print(f"\n\n✅ All {len(SCENARIOS)} scenarios processed")


  Healthy (all tests)
  Final: Healthy (confidence=0.9972)
  AD risk: 0.0139  |  PD risk: 0.0014
  AD best method: bayesian (models=['cdt', 'tmt', 'speech'], risk=0.0139)
  PD best method: bayesian (models=['spiral', 'meander', 'speech'], risk=0.0014)
  AD combos tested: 21
  PD combos tested: 21

  Clear AD
  Final: AD (confidence=0.9964)
  AD risk: 0.8570  |  PD risk: 0.0018
  AD best method: weighted_avg (models=['cdt', 'tmt', 'speech'], risk=0.8570)
  PD best method: bayesian (models=['spiral', 'meander', 'speech'], risk=0.0018)
  AD combos tested: 21
  PD combos tested: 21

  Clear PD
  Final: Healthy (confidence=0.9650)
  AD risk: 0.0175  |  PD risk: 0.0920
  AD best method: bayesian (models=['cdt', 'tmt', 'speech'], risk=0.0175)
  PD best method: bayesian (models=['spiral', 'meander', 'speech'], risk=0.0920)
  AD combos tested: 21
  PD combos tested: 21

  Borderline
  Final: Healthy (confidence=0.9752)
  AD risk: 0.0931  |  PD risk: 0.0124
  AD best method: bayesian (models=['

## Section 4: Compare Methods — Risk Score Heatmaps

Visualize how each method scores risk across all scenarios. This reveals:
- **Calibration differences**: Does Bayesian compress risk scores toward the prior?
- **Sensitivity**: Which method reacts most strongly to positive signals?
- **Specificity**: Which method best keeps healthy patients low-risk?

In [4]:
# ============================================================
# Section 4: Compare Methods — Risk Score Tables (All Models)
# ============================================================
# For each scenario, compare the "all models" combo across 3 methods.
# For partial data scenarios, use whatever combo is available.

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

methods = ["weighted_avg", "bayesian", "dempster_shafer"]
method_labels = ["Weighted Avg", "Bayesian", "Dempster-Shafer"]
scenarios_list = list(SCENARIOS.keys())

# Build AD heatmap: rows = scenarios, cols = methods
ad_matrix = np.zeros((len(scenarios_list), 3))
pd_matrix = np.zeros((len(scenarios_list), 3))

for i, name in enumerate(scenarios_list):
    r = all_results[name]
    # Group AD combos by method, take the "all" combo (or best available)
    for j, method in enumerate(methods):
        # Find the result with this method and the highest combo level
        candidates = [c for c in r.ad.all_combinations if c.method == method]
        if candidates:
            # Prefer "all" level, else highest model count
            best = sorted(candidates, key=lambda x: len(x.models_used), reverse=True)[0]
            ad_matrix[i, j] = best.risk
        else:
            ad_matrix[i, j] = np.nan
        
        candidates_pd = [c for c in r.pd.all_combinations if c.method == method]
        if candidates_pd:
            best_pd = sorted(candidates_pd, key=lambda x: len(x.models_used), reverse=True)[0]
            pd_matrix[i, j] = best_pd.risk
        else:
            pd_matrix[i, j] = np.nan

# --- Plot AD heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

cmap = plt.cm.RdYlGn_r  # Red = high risk, Green = low risk

for ax, matrix, title in [
    (axes[0], ad_matrix, "AD Risk Scores"),
    (axes[1], pd_matrix, "PD Risk Scores"),
]:
    im = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(3))
    ax.set_xticklabels(method_labels, fontsize=11, fontweight='bold')
    ax.set_yticks(range(len(scenarios_list)))
    ax.set_yticklabels(scenarios_list, fontsize=10)
    ax.set_title(title, fontsize=14, fontweight='bold', pad=10)
    
    # Annotate cells
    for si in range(matrix.shape[0]):
        for mi in range(matrix.shape[1]):
            val = matrix[si, mi]
            if not np.isnan(val):
                color = 'white' if val > 0.6 or val < 0.15 else 'black'
                ax.text(mi, si, f"{val:.3f}", ha='center', va='center',
                        fontsize=10, fontweight='bold', color=color)
            else:
                ax.text(mi, si, "N/A", ha='center', va='center',
                        fontsize=9, color='gray')
    
    # Add threshold line indication
    ax.axhline(y=-0.5, color='black', linewidth=2)

fig.colorbar(im, ax=axes, shrink=0.6, label='Risk Score (0=Healthy, 1=Disease)')
plt.suptitle("Fusion Method Comparison — Risk Scores (All Available Models)",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fusion_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Heatmap saved to fusion_heatmap.png")

ModuleNotFoundError: No module named 'matplotlib'

## Section 5: Combination Level Analysis

How does risk change as we add more models? We expect the "all models" combo to be more stable. Let's track risk progression: **Individual → Pairs → All** for each method.

In [ ]:
# ============================================================
# Section 5: Combination Level Analysis
# ============================================================
# Show how risk changes across combo levels for key scenarios

key_scenarios = ["Clear AD", "Clear PD", "Borderline", "Conflicting Models"]
tasks_to_show = [("ad", "AD"), ("pd", "PD")]

fig, axes = plt.subplots(len(key_scenarios), 2, figsize=(18, 4 * len(key_scenarios)))

colors = {"weighted_avg": "#2196F3", "bayesian": "#4CAF50", "dempster_shafer": "#FF9800"}
markers = {"weighted_avg": "o", "bayesian": "s", "dempster_shafer": "D"}

for row, scenario_name in enumerate(key_scenarios):
    r = all_results[scenario_name]
    
    for col, (task, task_label) in enumerate(tasks_to_show):
        ax = axes[row, col]
        task_result = r.ad if task == "ad" else r.pd
        
        for method in methods:
            method_combos = [c for c in task_result.all_combinations if c.method == method]
            if not method_combos:
                continue
            
            # Group by combo level
            level_order = {"individual": 1, "pair": 2, "triple": 3, "all": 4}
            by_level = defaultdict(list)
            for c in method_combos:
                by_level[c.combo_level].append(c.risk)
            
            x_vals, y_means, y_mins, y_maxs = [], [], [], []
            for level in sorted(by_level.keys(), key=lambda l: level_order.get(l, 5)):
                risks = by_level[level]
                x_vals.append(level_order.get(level, 5))
                y_means.append(np.mean(risks))
                y_mins.append(min(risks))
                y_maxs.append(max(risks))
            
            y_means = np.array(y_means)
            y_mins = np.array(y_mins)
            y_maxs = np.array(y_maxs)
            
            ax.plot(x_vals, y_means, color=colors[method], marker=markers[method],
                    linewidth=2, label=method.replace('_', ' ').title(), markersize=8)
            ax.fill_between(x_vals, y_mins, y_maxs, alpha=0.15, color=colors[method])
        
        ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
        ax.set_xlim(0.5, 4.5)
        ax.set_ylim(-0.05, 1.05)
        ax.set_xticks([1, 2, 3, 4])
        ax.set_xticklabels(["Individual", "Pair", "Triple", "All"], fontsize=9)
        ax.set_ylabel("Risk Score", fontsize=10)
        ax.set_title(f"{scenario_name} — {task_label}", fontsize=11, fontweight='bold')
        ax.legend(fontsize=8, loc='best')
        ax.grid(True, alpha=0.3)

plt.suptitle("Risk Score Progression Across Combination Levels\n(shaded = min/max range across combos)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fusion_combo_levels.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Combo level analysis saved to fusion_combo_levels.png")

## Section 6: Method Agreement & Consistency Analysis

A robust fusion method should be **internally consistent** — similar results regardless of which model combination is used. We measure:

1. **Cross-method agreement**: Do all 3 methods agree on the same risk level for a given combo?
2. **Intra-method stability**: Does a method give consistent results across different combos?
3. **Classification concordance**: Do all methods agree on the final label (AD/PD/Healthy)?

In [ ]:
# ============================================================
# Section 6: Method Agreement & Consistency Analysis
# ============================================================

print("=" * 80)
print("  CROSS-METHOD AGREEMENT (per scenario)")
print("=" * 80)

# For scenarios with all models, compare the 3 methods on the "all" combo
agreement_data = []

for name in scenarios_list:
    r = all_results[name]
    
    for task, task_result in [("AD", r.ad), ("PD", r.pd)]:
        # Get "all" combo for each method (or highest combo level)
        method_risks = {}
        for method in methods:
            candidates = [c for c in task_result.all_combinations if c.method == method]
            if candidates:
                best = sorted(candidates, key=lambda x: len(x.models_used), reverse=True)[0]
                method_risks[method] = best.risk
        
        if len(method_risks) >= 2:
            risks = list(method_risks.values())
            spread = max(risks) - min(risks)
            agreement = 1.0 - spread
            
            # Classification concordance
            labels = ["Positive" if r >= 0.5 else "Healthy" for r in risks]
            concordance = len(set(labels)) == 1
            
            agreement_data.append({
                "scenario": name,
                "task": task,
                "wa_risk": method_risks.get("weighted_avg", None),
                "bay_risk": method_risks.get("bayesian", None),
                "ds_risk": method_risks.get("dempster_shafer", None),
                "spread": spread,
                "agreement": agreement,
                "concordant": concordance,
            })

# Print table
print(f"\n{'Scenario':<30s} {'Task':>4s} {'WA':>7s} {'Bay':>7s} {'DS':>7s} "
      f"{'Spread':>7s} {'Agree':>6s} {'Same?':>5s}")
print("-" * 80)

for d in agreement_data:
    wa = f"{d['wa_risk']:.3f}" if d['wa_risk'] is not None else "  N/A"
    bay = f"{d['bay_risk']:.3f}" if d['bay_risk'] is not None else "  N/A"
    ds = f"{d['ds_risk']:.3f}" if d['ds_risk'] is not None else "  N/A"
    same = "✓" if d['concordant'] else "✗"
    print(f"{d['scenario']:<30s} {d['task']:>4s} {wa:>7s} {bay:>7s} {ds:>7s} "
          f"{d['spread']:>7.3f} {d['agreement']:>6.3f} {same:>5s}")

# Summary statistics
total = len(agreement_data)
concordant = sum(1 for d in agreement_data if d['concordant'])
avg_agreement = np.mean([d['agreement'] for d in agreement_data])
avg_spread = np.mean([d['spread'] for d in agreement_data])

print(f"\n{'─'*80}")
print(f"  Classification concordance: {concordant}/{total} ({100*concordant/total:.1f}%)")
print(f"  Average agreement score:    {avg_agreement:.3f}")
print(f"  Average risk spread:        {avg_spread:.3f}")

## Section 7: Intra-Method Stability

For each method, how much does the risk score vary across different model combinations for the same scenario? A stable method should give similar risk regardless of whether we use CDT alone or CDT+TMT+Speech together.

In [ ]:
# ============================================================
# Section 7: Intra-Method Stability (Variance across combos)
# ============================================================

print("=" * 80)
print("  INTRA-METHOD STABILITY")
print("  (Std dev of risk across all model combinations for each method)")
print("=" * 80)

stability_scores = {m: [] for m in methods}

# Only use full-data scenarios (at least 3 AD + 3 PD models)
full_scenarios = [name for name, scores in SCENARIOS.items() 
                  if sum(1 for k in scores if k.endswith('_ad')) >= 2 and
                     sum(1 for k in scores if k.endswith('_pd')) >= 2]

for name in full_scenarios:
    r = all_results[name]
    for task, task_result in [("AD", r.ad), ("PD", r.pd)]:
        for method in methods:
            risks = [c.risk for c in task_result.all_combinations if c.method == method]
            if len(risks) >= 2:
                stability_scores[method].append(np.std(risks))

print(f"\n{'Method':<20s} {'Mean Std':>10s} {'Max Std':>10s} {'Stability':>12s}")
print("-" * 55)

method_stability = {}
for method in methods:
    stds = stability_scores[method]
    if stds:
        mean_std = np.mean(stds)
        max_std = np.max(stds)
        # Stability = 1 - mean_std (higher = more stable)
        stability = 1.0 - mean_std
        method_stability[method] = stability
        print(f"{method:<20s} {mean_std:>10.4f} {max_std:>10.4f} {stability:>12.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
method_names = [m.replace('_', ' ').title() for m in methods]
stabilities = [method_stability.get(m, 0) for m in methods]
bar_colors = [colors[m] for m in methods]

bars = ax.bar(method_names, stabilities, color=bar_colors, edgecolor='black', linewidth=1.2)

for bar, val in zip(bars, stabilities):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{val:.4f}", ha='center', va='bottom', fontweight='bold', fontsize=12)

ax.set_ylabel("Stability Score (1 - mean σ)", fontsize=12)
ax.set_title("Intra-Method Stability\n(Higher = more consistent across model combinations)",
             fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig("fusion_stability.png", dpi=150, bbox_inches='tight')
plt.show()
print("\n✅ Stability chart saved to fusion_stability.png")

## Section 8: Correctness Scorecard

The ultimate test: **Does each method classify patients correctly?** We know the ground truth label for each scenario. Score each method on classification accuracy.

In [ ]:
# ============================================================
# Section 8: Correctness Scorecard
# ============================================================

# Ground truth labels for each scenario
GROUND_TRUTH = {
    "Healthy (all tests)":      "Healthy",
    "Clear AD":                 "AD",
    "Clear PD":                 "PD",
    "Borderline":               "Healthy",   # marginal = screen negative
    "Conflicting Models":       "AD",        # CDT strongly positive
    "Partial Data (CDT+Spiral)":"AD",        # CDT at 0.85
    "Single Model (CDT only)":  "AD",        # CDT at 0.82
    "Mixed AD+PD signals":      "AD",        # AD signals stronger
}

print("=" * 90)
print("  CORRECTNESS SCORECARD")
print("=" * 90)

# For each method, classify using fuse_quick and check
method_correct = {m: 0 for m in methods}
method_total = {m: 0 for m in methods}

print(f"\n{'Scenario':<30s} {'Truth':>8s} {'WA':>10s} {'Bay':>10s} {'DS':>10s}")
print("-" * 75)

for name, scores in SCENARIOS.items():
    truth = GROUND_TRUTH[name]
    row = f"{name:<30s} {truth:>8s}"
    
    for method in methods:
        quick = engine.fuse_quick(scores, method=method)
        pred = quick["classification"]
        correct = pred == truth
        method_total[method] += 1
        if correct:
            method_correct[method] += 1
        
        symbol = "✓" if correct else "✗"
        row += f" {pred:>7s} {symbol}"
    
    print(row)

print(f"\n{'─'*75}")
print(f"  {'ACCURACY':<30s} {'':>8s}", end="")
for method in methods:
    acc = method_correct[method] / method_total[method]
    print(f" {acc:>8.1%}  ", end="")
print()

# Also test with comprehensive fuse (robustness-selected best)
best_correct = 0
print(f"\n  {'Robustness-Selected Best':<30s} {'':>8s}", end="")
for name in SCENARIOS:
    r = all_results[name]
    if r.final_classification == GROUND_TRUTH[name]:
        best_correct += 1
best_acc = best_correct / len(SCENARIOS)
print(f" {best_acc:>8.1%}")

print(f"\n{'='*90}")
print("  KEY OBSERVATIONS:")
print("  • Weighted Average & Dempster-Shafer: raw risk > 0.5 ≈ Positive")
print("  • Bayesian: compressed by prior → often < 0.5 even for positive cases")
print("  • The 'best' selector uses robustness scoring to pick the most reliable combo")
print(f"{'='*90}")

## Section 9: Bayesian Threshold Optimization

The Bayesian method uses epidemiological priors which compress risk scores. A fixed 0.5 threshold may not be optimal. Let's find the **optimal threshold** for each method by sweeping across values and maximizing accuracy.

In [ ]:
# ============================================================
# Section 9: Threshold Sweep for Each Method
# ============================================================

thresholds = np.arange(0.01, 0.99, 0.01)

# For each method, compute accuracy at every threshold
# We use a simplified classification: for AD, risk >= threshold → AD detected

# Build ground truth labels per task
ad_truth = {}  # scenario → True if AD
pd_truth = {}  # scenario → True if PD
for name in SCENARIOS:
    gt = GROUND_TRUTH[name]
    ad_truth[name] = gt == "AD"
    pd_truth[name] = gt == "PD"

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, task, truth_dict, task_label in [
    (axes[0], "ad", ad_truth, "AD Detection"),
    (axes[1], "pd", pd_truth, "PD Detection"),
]:
    for method in methods:
        accuracies = []
        for thresh in thresholds:
            correct = 0
            total = 0
            for name, scores in SCENARIOS.items():
                quick = engine.fuse_quick(scores, method=method)
                risk = quick[f"{task}_risk"]
                predicted = risk >= thresh
                actual = truth_dict[name]
                if predicted == actual:
                    correct += 1
                total += 1
            accuracies.append(correct / total)
        
        accuracies = np.array(accuracies)
        best_idx = np.argmax(accuracies)
        best_thresh = thresholds[best_idx]
        best_acc = accuracies[best_idx]
        
        ax.plot(thresholds, accuracies, color=colors[method], linewidth=2,
                label=f"{method.replace('_', ' ').title()} (best={best_thresh:.2f}, acc={best_acc:.0%})")
        ax.axvline(x=best_thresh, color=colors[method], linestyle=':', alpha=0.5)
    
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.6, label='Default (0.5)')
    ax.set_xlabel("Threshold", fontsize=12)
    ax.set_ylabel("Accuracy", fontsize=12)
    ax.set_title(f"{task_label} — Threshold Sweep", fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.suptitle("Optimal Threshold per Method\n(Sweep across 0.01→0.99)",
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("fusion_threshold_sweep.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Threshold sweep saved to fusion_threshold_sweep.png")

## Section 10: Robustness Ranking — Final Recommendation

Score each method on a **composite metric** combining:
- **Accuracy** (with optimal threshold)
- **Stability** (low variance across combos)
- **Agreement** (concordance with other methods)
- **Calibration** (healthy → low risk, disease → high risk)

Then produce the final recommendation for wiring into the backend.

In [ ]:
# ============================================================
# Section 10: Final Robustness Ranking
# ============================================================

print("=" * 80)
print("  FINAL METHOD RANKING")
print("=" * 80)

# --- 1. Accuracy with optimal thresholds ---
def compute_best_accuracy(method):
    """Find best accuracy across all thresholds for overall classification."""
    best = 0
    best_t_ad = 0.5
    best_t_pd = 0.5
    
    for t_ad in np.arange(0.05, 0.95, 0.05):
        for t_pd in np.arange(0.05, 0.95, 0.05):
            correct = 0
            for name, scores in SCENARIOS.items():
                quick = engine.fuse_quick(scores, method=method)
                ad_r, pd_r = quick["ad_risk"], quick["pd_risk"]
                
                if ad_r >= t_ad and ad_r >= pd_r:
                    pred = "AD"
                elif pd_r >= t_pd and pd_r > ad_r:
                    pred = "PD"
                else:
                    pred = "Healthy"
                
                if pred == GROUND_TRUTH[name]:
                    correct += 1
            
            acc = correct / len(SCENARIOS)
            if acc > best:
                best = acc
                best_t_ad = t_ad
                best_t_pd = t_pd
    
    return best, best_t_ad, best_t_pd

method_metrics = {}

for method in methods:
    acc, opt_ad_t, opt_pd_t = compute_best_accuracy(method)
    stab = method_stability.get(method, 0)
    
    # Calibration: measure correlation between risk and ground truth
    cal_scores = []
    for name, scores in SCENARIOS.items():
        quick = engine.fuse_quick(scores, method=method)
        # True AD should have high ad_risk, etc.
        gt = GROUND_TRUTH[name]
        if gt == "AD":
            cal_scores.append(quick["ad_risk"])  # higher = better
        elif gt == "PD":
            cal_scores.append(quick["pd_risk"])  # higher = better
        else:
            # Healthy: both should be low
            cal_scores.append(1.0 - max(quick["ad_risk"], quick["pd_risk"]))
    
    calibration = np.mean(cal_scores)
    
    # Composite score
    composite = 0.35 * acc + 0.25 * stab + 0.20 * calibration + 0.20 * (1.0 - avg_spread)
    
    method_metrics[method] = {
        "accuracy": acc,
        "optimal_ad_threshold": opt_ad_t,
        "optimal_pd_threshold": opt_pd_t,
        "stability": stab,
        "calibration": calibration,
        "composite": composite,
    }

# Print ranking
print(f"\n{'Method':<22s} {'Accuracy':>10s} {'Stability':>10s} {'Calibr':>10s} "
      f"{'Composite':>10s} {'AD thresh':>10s} {'PD thresh':>10s}")
print("-" * 85)

ranked = sorted(method_metrics.items(), key=lambda x: x[1]['composite'], reverse=True)

for rank, (method, m) in enumerate(ranked, 1):
    marker = " ⭐ BEST" if rank == 1 else ""
    print(f"{method:<22s} {m['accuracy']:>10.1%} {m['stability']:>10.4f} "
          f"{m['calibration']:>10.4f} {m['composite']:>10.4f} "
          f"{m['optimal_ad_threshold']:>10.2f} {m['optimal_pd_threshold']:>10.2f}{marker}")

# --- Radar chart ---
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

categories = ['Accuracy', 'Stability', 'Calibration', 'Simplicity']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

# Simplicity: WA=1.0, Bayesian=0.7, DS=0.5
simplicity = {"weighted_avg": 1.0, "bayesian": 0.7, "dempster_shafer": 0.5}

for method in methods:
    m = method_metrics[method]
    values = [m['accuracy'], m['stability'], m['calibration'], simplicity[method]]
    values += values[:1]
    
    ax.plot(angles, values, color=colors[method], linewidth=2, linestyle='-',
            label=method.replace('_', ' ').title())
    ax.fill(angles, values, color=colors[method], alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.set_title("Method Comparison Radar", fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)

plt.tight_layout()
plt.savefig("fusion_radar.png", dpi=150, bbox_inches='tight')
plt.show()

# Final recommendation
best_method, best_metrics = ranked[0]
print(f"\n{'='*80}")
print(f"  🏆 RECOMMENDED METHOD: {best_method.replace('_', ' ').upper()}")
print(f"{'='*80}")
print(f"  Composite score: {best_metrics['composite']:.4f}")
print(f"  Accuracy:        {best_metrics['accuracy']:.1%}")
print(f"  Stability:       {best_metrics['stability']:.4f}")
print(f"  Calibration:     {best_metrics['calibration']:.4f}")
print(f"  Optimal AD threshold: {best_metrics['optimal_ad_threshold']:.2f}")
print(f"  Optimal PD threshold: {best_metrics['optimal_pd_threshold']:.2f}")
print(f"{'='*80}")

## Section 11: Export Configuration for Backend Wiring

Export the winning method's configuration as a JSON file that the backend's `NeuroVerseFusionEngine` can load directly via `config_path`.

In [ ]:
# ============================================================
# Section 11: Export Configuration for Backend
# ============================================================

best_method_name = ranked[0][0]
best_m = ranked[0][1]

config = {
    "recommended_method": best_method_name,
    "model_aucs": MODEL_AUCS,
    "prior_ad": PRIOR_AD,
    "prior_pd": PRIOR_PD,
    "ad_threshold": float(best_m["optimal_ad_threshold"]),
    "pd_threshold": float(best_m["optimal_pd_threshold"]),
    "evaluation": {
        "scenarios_tested": len(SCENARIOS),
        "accuracy": float(best_m["accuracy"]),
        "stability": float(best_m["stability"]),
        "calibration": float(best_m["calibration"]),
        "composite_score": float(best_m["composite"]),
    },
    "all_methods_ranked": [
        {
            "method": method,
            "composite": float(metrics["composite"]),
            "accuracy": float(metrics["accuracy"]),
            "optimal_ad_threshold": float(metrics["optimal_ad_threshold"]),
            "optimal_pd_threshold": float(metrics["optimal_pd_threshold"]),
        }
        for method, metrics in ranked
    ],
}

config_path = "fusion_config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Configuration exported to {config_path}")
print(f"\nTo wire into backend:")
print(f"  engine = NeuroVerseFusionEngine(config_path='{config_path}')")
print(f"  result = engine.fuse_quick(scores, method='{best_method_name}')")
print(f"\n--- Exported Config ---")
print(json.dumps(config, indent=2))

## Section 12: Validate Wired Connections

Quick sanity check — load the exported config back into a fresh engine and verify it produces the same results.

In [ ]:
# ============================================================
# Section 12: Validate Wired Config
# ============================================================

# Load a fresh engine from exported config
engine2 = NeuroVerseFusionEngine(config_path=config_path)

print("Validation: Comparing original engine vs config-loaded engine\n")

all_pass = True
for name, scores in SCENARIOS.items():
    r1 = engine.fuse_quick(scores, method=best_method_name)
    r2 = engine2.fuse_quick(scores, method=best_method_name)
    
    match_ad = abs(r1["ad_risk"] - r2["ad_risk"]) < 1e-6
    match_pd = abs(r1["pd_risk"] - r2["pd_risk"]) < 1e-6
    match_cls = r1["classification"] == r2["classification"]
    
    ok = match_ad and match_pd and match_cls
    if not ok:
        all_pass = False
    
    status = "✓ PASS" if ok else "✗ FAIL"
    print(f"  {status}  {name:<30s}  AD={r2['ad_risk']:.4f}  PD={r2['pd_risk']:.4f}  "
          f"Class={r2['classification']}")

print(f"\n{'='*60}")
if all_pass:
    print("  ✅ ALL VALIDATIONS PASSED — Config is ready for backend wiring!")
else:
    print("  ⚠️  SOME VALIDATIONS FAILED — Check config export")
print(f"{'='*60}")

# Print backend integration snippet
print(f"""
╔══════════════════════════════════════════════════════════╗
║  BACKEND INTEGRATION (copy to test_service.py)          ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  from app.ml.fusion import get_fusion_engine             ║
║                                                          ║
║  engine = get_fusion_engine("fusion_config.json")        ║
║  result = engine.fuse(model_scores)                      ║
║  # or for quick API:                                     ║
║  quick = engine.fuse_quick(scores, "{best_method_name}")  ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝
""")